In [ ]:
# NCrystal notebook settings
# title: Example: Compute in-scattering
# menutitle: In-scattering
# shortkey: nw12inscatter
# section: neuwave12
# requires: plot, tqdm
# test-parameters: N = 100000
# slow: yes

## Introduction [intro]
In this example we will compute the in-scattering probability using different techniques, and see its effect on the wavelength-dependent transmission.

### Additional modules: [modules]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tqdm.notebook

## Example 1 [example1]
Compute the in-scattering probability for a graphite sample, with a detector that covers 10 degree from the beam axis.

For a first attempt, we can use the function `sampleScatterIsotropic()` with the `repeat` argument, to sample multiple times a scattering event with a fixed incident energy:

In [ ]:
thickness = 1.0 # cm
max_theta = 10 # deg

In [ ]:
cfg = 'C_sg194_pyrolytic_graphite.ncmat'
mat = NC.load(cfg)

In [ ]:
#Number of sampled scattering events for each wavelength (reduce it for faster,
#but noisier, results):
N = 1000000

In [ ]:
wl1 = np.linspace(1,8,351)
P_non_absorption = np.exp(-thickness*mat.info.getNumberDensity()*mat.absorption.xsect(wl=wl1))
P_non_scatttering = np.exp(-thickness*mat.info.getNumberDensity()*mat.scatter.xsect(wl=wl1))
P_forward = []
min_mu = np.cos(max_theta*np.pi/180.0)
for wl in tqdm.notebook.tqdm(wl1, total=len(wl1)):
  E_vals, mu_vals = mat.scatter.sampleScatterIsotropic(ekin=NC.wl2ekin(wl), repeat=N)
  forward_count = (mu_vals > min_mu).sum()
  P_forward.append(forward_count/N)

We compare this with the theoretical transmission computed from the macroscopic total cross section:

In [ ]:
wl0 = np.linspace(1,8,1000)
xs = (mat.scatter.xsect(wl=wl0)+mat.absorption.xsect(wl=wl0))
dens = mat.info.getNumberDensity()
transmission0 = np.exp(-dens*xs*thickness)

In [ ]:
plt.figure()
plt.plot(wl0, transmission0, 'k', label = 'Theoretical transmission')
plt.plot(wl1, P_non_absorption*P_non_scatttering, 'r--', label = 'Uncollided')
plt.plot(wl1, P_non_absorption*P_non_scatttering+P_forward, 'b-', label = 'Uncollided + forw. scatt.')
plt.legend()
plt.xlabel('Wavelength [A]')
plt.ylabel('Transmission')

The effect is actually very small. We can check by computing the probability near the longest Bragg edge:

In [ ]:
wl = 6.7 # A
E_vals, mu_vals = mat.scatter.sampleScatterIsotropic(ekin=NC.wl2ekin(wl), repeat=N)
theta_vals = np.arccos(mu_vals)*180.0/np.pi
count = (theta_vals < max_theta).sum()
print(f' The probability of forward scattering below {max_theta:.1f} deg is {count/N:.3e}')

and plot the distribution:

In [ ]:
theta_bins = np.linspace(0,180,181)
plt.figure()
a = plt.hist(theta_vals,bins=theta_bins, density=False)
plt.yscale('log')
plt.xlim(-1,180)
plt.xlabel('Angle [deg]')
plt.ylabel('Counts per bin')

We will now repeat the exercise but using the mini Monte Carlo included in NCrystal

In [ ]:
import NCrystal.minimc as ncmmc
result = ncmmc.run(cfg,scenario=f'2Aa pencil on {thickness}cm sphere')
result.tally('theta').plot(logy=True)

The `MMCResults` class includes all details about both configuration of the simulation, as well as outcomes. In particular specific tallies can be access via the `.tally` function as we saw. For more details, see the dedicated notebook on the MiniMC and the wiki at https://github.com/mctools/ncrystal/wiki/minimc.

For the present purposes, we want to access the `theta` (scattering angle) tally, and get both the total histogram, as well as the histogram with the unscattered events. We also note that the histogram themselves are instances of the `Hist1D` class, which includes several useful methods. We will use `Hist1D.integrate()` to get the number of counts in a range.

In [ ]:
help(result.tally('theta').hist_total)

Finally `MMCResults.output_metadata` contains number of simulated particles that will be used for normalisation:

In [ ]:
result.output_metadata

We can put these things together to compute the in-scattering probability including multiple scattering effects:

In [ ]:
wl2 = np.linspace(1,8,700)
transmission2a = []
transmission2b = []

for wl in tqdm.notebook.tqdm(wl2, total=len(wl2)):
  result = ncmmc.run(cfg,scenario=f'{wl}Aa pencil on {thickness}cm sphere 1e4 times')
  nstat=result.output_metadata['provided']['count']  
  t = result.tally('theta')
  transmission2b.append(t.hist_total.integrate(0,max_theta)[0]/nstat)
  transmission2a.append(t.hist_breakdown['NOSCAT'].integrate(0,max_theta)[0]/nstat)

In [ ]:
plt.figure()
plt.plot(wl0, transmission0, 'k', label = 'Theoretical transmission')
plt.plot(wl2, transmission2a, 'r--', label = 'Uncollided')
plt.plot(wl2, transmission2b, 'b-', label = 'Uncollided + forw. scatt.')
plt.legend()
plt.xlabel('Wavelength [A]')
plt.ylabel('Transmission')

We can look more in detail why there is a significant in-scattering contribution near the $\lambda = 6.7$ Å Bragg edge:

In [ ]:
result = ncmmc.run(cfg,scenario=f'{6.7}Aa pencil on {thickness}cm sphere')

result.tally('theta').plot()

compared with the Bragg edge near $\lambda = 4$ Å:

In [ ]:
result = ncmmc.run(cfg,scenario=f'4.06Aa pencil on {thickness}cm')

result.tally('theta').plot()

As we can see, the isolated Bragg edge at $\lambda = 6.7$ Å has a larger probability of two scattering events near 180 degrees, which send the neutrons back in the forward direction. Whereas in for $\lambda = 4.06$ Å three Bragg edges compete and send neutrons in different directions in multiple scattering events, with a smaller relative contribution in the forward direction.

## Exercise [exercise]
Repeat the calculation with light water and polyethylene.